# Coin Extraction

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [76]:
import numpy as np

from skimage import filters, feature, data
from skimage.morphology import opening, closing, disk


from scripts.preprocessing import (scale_hough_circles, get_hough_circles, read_image, apply_hsv_threshold,
                                   adjust_contrast, apply_sobel, apply_hough, apply_closing, apply_opening, apply_operation, apply_black_tophat, apply_white_tophat, apply_erosion, apply_dilation, remove_holes)
from scripts.viz import plot_images
from scripts.config import EXAMPLE_IMGS, NOISY_BG_OUTLIERS, HAND_IMGS, PROBLEMATIC_HOUGH_CIRCLES

## Parameters

In [ ]:
operation_dict = {'opening': apply_opening,
	'sobel': apply_sobel,
	'threshold': apply_hsv_threshold,
	'contrast': adjust_contrast,
	'erosion': apply_erosion,
	'dilation': apply_dilation,
	'blackth': apply_black_tophat,
	'whiteth': apply_white_tophat
	}

## Pipeline

In [ ]:
coin_template = data.coins()[170:220, 75:130]
coin_template = filters.sobel(coin_template)

import numpy as np
from matplotlib import pyplot as plt


for file in EXAMPLE_IMGS:

    img = read_image(file)

    disk_radius = 10
    disk_morph = disk(disk_radius)

    # morphology
    img_open = np.zeros_like(img)

    for channel in range(img.shape[2]):
        img_open[:,:,channel] = opening(img[:,:,channel], disk_morph)
   
    contrasted = adjust_contrast(img_open)
    thresholded = apply_hsv_threshold(contrasted.copy())
    sobel = apply_sobel(thresholded)

    heatmap_sobel = feature.match_template(sobel, coin_template)
    hough = apply_hough(sobel, img)
    
    plot_images(
        original=img,
        enhanced=contrasted,
        thresholded=thresholded,
        sobel=sobel,
        hough=hough
    )
    

In [ ]:
coin_template = data.coins()[170:220, 75:130]
coin_template = filters.sobel(coin_template)

for file in EXAMPLE_IMGS:

    img = read_image(file)

    # morphology
    img_open = np.zeros_like(img)

    for channel in range(img.shape[2]):
        img_open[:,:,channel] = opening(img[:,:,channel], disk_morph)
   
    contrasted = adjust_contrast(img_open)
    #contrasted = adjust_contrast(img)
    thresholded = apply_hsv_threshold(contrasted.copy())
    sobel = apply_sobel(thresholded)

    heatmap_sobel = feature.match_template(sobel, coin_template)
    hough = apply_hough(sobel, img)
    
    plot_images(
        original=img,
        enhanced=contrasted,
        thresholded=thresholded,
        sobel=sobel,
        hough=hough
    )
    
    """plot_colors_histo(
        img=enhanced,
        func=extract_hsv_channels,
        labels=["Hue", "Saturation", "Value"],
    )"""

Let's test it on images with a hand.

In [ ]:
for file in HAND_IMGS:

    img = read_image(file)

    # morphology
    
    disk_radius = 15
    disk_morph = disk(disk_radius)
    img_open = np.zeros_like(img)

    for channel in range(img.shape[2]):
        img_open[:,:,channel] = closing(img[:,:,channel], disk_morph)
   
    contrasted_morph = adjust_contrast(img_open)
    thresholded_morph = apply_hsv_threshold(contrasted_morph.copy())
    sobel_morph = apply_sobel(thresholded_morph)
    houghmorph = apply_hough(sobel_morph, img)
    
    contrasted = adjust_contrast(img)
    thresholded = apply_hsv_threshold(contrasted.copy())
    sobel = apply_sobel(thresholded)
    hough = apply_hough(sobel, img)
    
    plot_images(
        original=img,
        enhanced=contrasted_morph,
        thresholded=thresholded_morph,
        hough=hough,
        houghmorph=houghmorph
    )
    #break

Let's see how it works with noisy background and OOD.

## List of problematic pictures 

coins are missing (some coins are not marked with Hough circles):

- L1010475.JPG
- L1010478.JPG
- L1010479.JPG
- L1010485.JPG
- L1010486.JPG
- L1010491.JPG
- L1010500.JPG
- L1010503.JPG



# Permutations on problematic pictures

Orders tried :
1. ['opening', 'contrast', 'threshold', 'sobel'] doesn't catch all coins but has no false positive
2. ['contrast', 'opening', 'threshold', 'sobel'] performs better than 1 but not all coins are caught
3. ['contrast', 'threshold', 'opening', 'sobel'] completely fails to find coins
4. ['opening', 'threshold', 'contrast', 'sobel'] has bad performances
5. ['threshold', 'opening', 'contrast', 'sobel'] has the second worst performances
6. ['threshold', 'contrast', 'opening', 'sobel'] has the worst performances

Overall, ['contrast', 'opening', 'threshold', 'sobel'] has the best performances although it's not failproof on the `PROBLEMATIC_HOUGH_CIRCLES` dataset.

In [ ]:
operation_list = ['threshold', 'contrast', 'opening', 'sobel']

for file in PROBLEMATIC_HOUGH_CIRCLES:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )


# Permutations on noisy BG

Orders tried :


In [ ]:
operation_list = ['contrast', 'opening', 'threshold', 'sobel']

for file in NOISY_BG_OUTLIERS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )


# Full pipeline shown on hand images

## Showcase of detected circles

In [ ]:
operation_list = ['contrast', 'opening', 'threshold', 'sobel']

for file in HAND_IMGS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )


## Return Hough circles only

In [ ]:
operation_list = ['contrast', 'opening', 'threshold', 'sobel']

hough_circles_list = []

for file in HAND_IMGS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = get_hough_circles(output_list[-1], img)

    hough_circles_list.append(hough)

hough_circles = np.concatenate(([i for i in hough_circles_list]))


In [82]:
hough_circles[0:3]

array([[314, 260,  37],
       [282, 186,  32],
       [356, 194,  29]], dtype=uint16)

## Rescale Hough circles to match original picture size

In [87]:
scaled = scale_hough_circles(hough_circles, 10, 6000, 4000)
scaled[0:3]

array([[3140, 2600,  370],
       [2820, 1860,  320],
       [3560, 1940,  290]])

# Other morphological tools

## Black Tophat

Tried:

1. ['contrast', 'blackth', 'threshold', 'sobel'] bad results
2.  ['contrast', 'opening', 'blackth', 'threshold', 'sobel'] bad results
3. ['blackth', 'threshold', 'sobel'] bad results

Overall, nothing of interest

In [ ]:
operation_list = ['blackth', 'threshold', 'sobel']

for file in NOISY_BG_OUTLIERS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )


## White Tophat

Tried:

1. ['contrast', 'whiteth', 'threshold', 'sobel'] bad results
2. ['threshold', 'whiteth', 'sobel'] bad results

Tophat is probably not useful for this application

In [ ]:
operation_list = ['contrast', 'opening', 'threshold', 'whiteth', 'sobel']

for file in NOISY_BG_OUTLIERS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )


## Misc

Various tries with dilation, erosion and other tools from scikit-image morphology.

In [ ]:
img = read_image(HAND_IMGS[0])

In [ ]:
img_removeholes = remove_holes(np.copy(img[:,:,0]), 1)

In [ ]:
print(img_removeholes.shape)


plt.imshow(img_removeholes)

In [ ]:
operation_list = ['contrast', 'removeholes', 'threshold','sobel']

for file in HAND_IMGS:

    img = read_image(file)
    print(f"Treating img {file}")

    # Applying all operations but Hough transform
    output_list = apply_operation(img, operation_list, operation_dict)

    # Hough transform on last item and add to list
    hough = apply_hough(output_list[-1], img)

    
    output_list.append(hough)
    
    # Zip for plot
    img_dict = dict(zip(operation_list + ['hough'], output_list))

    plot_images(
        **img_dict
    )
